# 1.3 Levels of Measurement

**Goal:** determine which comparisons and numerical summaries are justified by what a variable measures.

**Prerequisites:** [Types of Data](2_Types-of-Data.ipynb). **Runtime:** Python 3.10+, standard library only. Saved outputs are included; run cells in order. Examples are synthetic.

## Contents
1. Meaning of a measurement scale
2. Nominal scale
3. Ordinal scale
4. Interval scale
5. Ratio scale
6. Comparison table and preprocessing example
7. Practice and solutions

## 1. What does a measurement scale tell us?

A measurement scale describes the information represented by values and which transformations preserve that information. It is about meaning, not a Python dtype. Assigning numbers to labels does not create quantitative distances.

Four questions help classify a field:

1. Do values distinguish categories?
2. Is there a meaningful order?
3. Are equal numerical differences meaningful and comparable?
4. Is zero a meaningful absence of the measured amount, so ratios are meaningful?

These lead to nominal, ordinal, interval, and ratio scales. Real applications can be ambiguous: document the construct, instrument, and assumptions rather than classifying by column name alone.

## 2. Nominal Scale

Nominal values identify **unordered categories**: browser, country, blood group, or product type. Only equality and inequality of category identity are intrinsic. Numeric codes such as 1=Firefox and 2=Chrome are still nominal.

**Permissible transformation:** any one-to-one relabeling. Replacing category codes 1 and 2 with 100 and 7 must not change the substantive information.

**Useful summaries:** counts, proportions, and modes. There may be multiple tied modes. A mean, median, or subtraction of arbitrary category codes is not a meaningful description of category identity.

**Worked example:** browser labels A, A, B, C give proportions 0.50, 0.25, 0.25. Relabeling A as 99 changes no category frequency but radically changes the mean of codes.

**ML implication:** one-hot encoding can represent unordered categories without imposing a numerical order. Decide how to handle unseen and missing categories; learn the vocabulary from training data. High-cardinality identifiers need special scrutiny because memorization can replace generalization.

In [1]:
from collections import Counter
import statistics

labels = ["A", "A", "B", "C"]
counts = Counter(labels)
print("Category proportions:", {label: count/len(labels) for label, count in sorted(counts.items())})
encoding_1 = {"A": 1, "B": 2, "C": 3}
encoding_2 = {"A": 99, "B": 2, "C": 3}
print("Arbitrary code means:", statistics.mean(encoding_1[x] for x in labels),
      statistics.mean(encoding_2[x] for x in labels))
print("The categories and frequencies are identical under both encodings.")

Category proportions: {'A': 0.5, 'B': 0.25, 'C': 0.25}
Arbitrary code means: 1.75 50.75
The categories and frequencies are identical under both encodings.


## 3. Ordinal Scale

Ordinal values have a meaningful order but not guaranteed equal gaps: low/medium/high risk, beginner/intermediate/advanced skill, or a single 1–5 satisfaction item. “High” is above “medium,” but the gap need not equal the gap between “medium” and “low.”

**Permissible transformation:** any strictly increasing recoding preserves the ordering. Codes $1,2,3$ and $1,10,100$ describe the same order but produce different numerical averages and spacings.

**Useful summaries:** ordered frequencies, cumulative proportions, modes, and order-based quantiles. For an ordinal variable, report a median category or central categories; do not invent a category by averaging two middle codes. Rank-based analyses can respect order without assuming equally spaced levels.

**Worked example:** ratings poor, fair, good, good, excellent have median category good. An ordinal encoding can help a model access order, but a linear model treating codes as numeric imposes a fixed effect per code unit. That is an extra assumption, not a property guaranteed by the rating scale.

**Rating caution:** a single Likert-type item is ordinarily treated as ordinal. A validated composite of several items is sometimes modeled approximately as interval under explicit assumptions. The scale classification alone does not settle every modeling decision.

In [2]:
order = {"poor": 0, "fair": 1, "good": 2, "excellent": 3}
ratings = ["good", "poor", "excellent", "fair", "good"]
ordered = sorted(ratings, key=order.get)
print("Ordered categories:", ordered)
print("Median category:", ordered[len(ordered)//2])
print("Proportion at or below fair:", sum(order[x] <= order["fair"] for x in ratings)/len(ratings))
print("Two valid order-preserving code means:",
      statistics.mean([1, 2, 3]), statistics.mean([1, 10, 100]))

Ordered categories: ['poor', 'fair', 'good', 'good', 'excellent']
Median category: good
Proportion at or below fair: 0.4
Two valid order-preserving code means: 2 37


## 4. Interval Scale

An interval scale has meaningful order and equal differences, but its zero is an arbitrary reference rather than absence of the quantity. Celsius and Fahrenheit temperature are common examples; calendar time on a specified uniform unit convention is another.

**Permissible transformation:** $y=ax+b$ with $a>0$. This changes unit size and origin. Differences transform consistently: $y_2-y_1=a(x_2-x_1)$. Ratios of raw values generally change when $b\ne0$.

**Worked example:** 10°C to 20°C is a rise of 10 Celsius degrees, the same difference as 20°C to 30°C. But 20°C is not twice as hot as 10°C on a temperature-ratio scale. Converting with $F=1.8C+32$ gives 50°F and 68°F, whose ratio is 1.36 rather than 2.

**Useful summaries:** means, medians, differences, variance, and standard deviation when the substantive question supports them. Standard deviation changes by factor $a$; variance by $a^2$. A coefficient of variation (SD/mean) or a “percent increase” of raw values is generally not meaningful with an arbitrary zero.

Intervals between timestamps can be ratio-scale durations, even though the timestamps themselves use an arbitrary origin. Distinguish a date from elapsed time.

In [3]:
celsius = [10, 20, 30]
fahrenheit = [1.8*c + 32 for c in celsius]
print("Celsius:", celsius, "Fahrenheit:", fahrenheit)
print("Raw-value ratio changes:", celsius[1]/celsius[0], fahrenheit[1]/fahrenheit[0])
print("Mean conversion:", statistics.mean(fahrenheit), 1.8*statistics.mean(celsius)+32)
print("SD conversion:", statistics.pstdev(fahrenheit), 1.8*statistics.pstdev(celsius))

Celsius: [10, 20, 30] Fahrenheit: [50.0, 68.0, 86.0]
Raw-value ratio changes: 2.0 1.36
Mean conversion: 68.0 68.0
SD conversion: 14.696938456699069 14.696938456699069


## 5. Ratio Scale

A ratio scale has meaningful order, equal differences, and a meaningful zero for the measured amount. Examples include length, mass, elapsed duration, and counts. Absolute thermodynamic temperature measured in Kelvin is a ratio-scale example: 0 K is absolute zero; this does not mean all conceivable physical energy is absent.

**Permissible transformation:** $y=ax$ with $a>0$, a change of unit without a shifted zero. Ratios are invariant: $y_2/y_1=x_2/x_1$ when the denominator is nonzero.

**Worked example:** a 10-second duration is twice a 5-second duration. In milliseconds the values are 10,000 and 5,000; the ratio remains 2. A zero-second elapsed duration means none of that duration elapsed.

**Useful summaries:** appropriate means, quantiles, variance/SD, and ratios. Coefficients of variation are often used for positive ratio-scale quantities with a nonzero mean. Geometric means require positive values in their usual definition; a ratio scale does not remove formula-specific domain restrictions. Division by zero remains undefined.

Income needs context: gross nonnegative amounts can support ratio interpretations, while net balances with debts, arbitrary indexes, or scores called “income” may require different reasoning. Zero and units must be defined for the actual variable.

In [4]:
seconds = [5, 10, 15]
milliseconds = [1000*x for x in seconds]
print("Ratio in seconds:", seconds[1]/seconds[0])
print("Ratio in milliseconds:", milliseconds[1]/milliseconds[0])
cv_seconds = statistics.pstdev(seconds)/statistics.mean(seconds)
cv_milliseconds = statistics.pstdev(milliseconds)/statistics.mean(milliseconds)
print("CV in both units:", cv_seconds, cv_milliseconds)
assert abs(cv_seconds-cv_milliseconds) < 1e-12

Ratio in seconds: 2.0
Ratio in milliseconds: 2.0
CV in both units: 0.408248290463863 0.408248290463863


## 6. Compare the four scales

| Scale | Category identity | Order | Equal differences | Meaningful zero/ratios | Example |
|---|---|---|---|---|---|
| Nominal | Yes | No intrinsic order | No | No | Device OS |
| Ordinal | Yes | Yes | Not guaranteed | No | Risk category |
| Interval | Yes | Yes | Yes | No | Celsius temperature |
| Ratio | Yes | Yes | Yes | Yes | Elapsed seconds |

This is a guide to defensible interpretation, not an automatic algorithm selector. Distribution shape, the study question, repeated observations, missingness, and model assumptions also matter. Numeric encoding cannot upgrade a measurement scale.

### Worked preprocessing example

Suppose a training dataset contains device OS, satisfaction category, temperature in Celsius, and elapsed duration. A possible plan is to one-hot encode OS, preserve satisfaction order with an explicitly documented assumption, and scale quantitative columns using **training-only** summaries. Learn category vocabularies and imputation rules on training data as well.

The code demonstrates only nominal encoding and numeric scaling. Population SD (`pstdev`) here describes the training values used for scaling; it is not being used as an unbiased estimator of a population variance. A standardized ratio-scale variable no longer has its original meaningful zero, so ratios of standardized values are not original quantity ratios.

In [5]:
train_os = ["android", "ios", "android"]
vocabulary = sorted(set(train_os))
def encode_os(value):
    if value not in vocabulary:
        raise ValueError("Unknown category: handle deliberately before modeling")
    return [int(value == category) for category in vocabulary]
print("Training vocabulary:", vocabulary)
print("One-hot rows:", [encode_os(value) for value in train_os])
train_duration = [5.0, 10.0, 15.0]
train_mean = statistics.mean(train_duration)
train_sd = statistics.pstdev(train_duration)
assert train_sd > 0
held_out_duration = 20.0
print("Held-out standardized value:", (held_out_duration-train_mean)/train_sd)
print("Training mean/SD were not recomputed using the held-out value.")

Training vocabulary: ['android', 'ios']
One-hot rows: [[1, 0], [0, 1], [1, 0]]
Held-out standardized value: 2.449489742783178
Training mean/SD were not recomputed using the held-out value.


## Practice

1. Classify blood group, finishing place in a race, Celsius temperature, and elapsed race time.
2. A satisfaction scale is recoded from 1,2,3 to 1,4,100. Which measurement information is preserved?
3. Explain why a mean of postal codes is not a meaningful location summary.
4. Is a 10°C reading half a 20°C reading in a unit-independent temperature-ratio sense?
5. A timestamp records seconds since a chosen epoch. How does its scale differ from a duration between two timestamps?
6. Why can ratios of z-scores fail to reflect ratios of original heights?
7. How should you report the center of four ordered categories: poor, fair, good, excellent?
8. Does a ratio scale guarantee a geometric mean is valid when some values are zero?

## Solutions

1. Nominal, ordinal, interval, ratio, respectively.
2. Order and identity are preserved; equal numerical gaps are not. Means of these codes need an extra equal-spacing interpretation to be meaningful.
3. Postal codes label areas; arbitrary numeric coding does not measure distance or quantity. Use actual geography for location summaries.
4. No. Converting to Fahrenheit shifts the origin and changes the raw-value ratio; the difference remains meaningful.
5. The timestamp has an arbitrary origin (interval interpretation); elapsed duration has a meaningful zero and supports duration ratios.
6. Standardization subtracts the mean, shifting zero. Ratios after that shift are not invariant ratios of the original height measurements.
7. The middle categories are fair and good. Report those central categories or state an explicit ordinal-median convention rather than inventing a halfway category.
8. No. The usual geometric mean/log formulation assumes positive values. Formula domains still apply, and zeros need deliberate treatment.

**Readiness check:** explain why “stored as numbers” does not imply “safe to average,” and why a change of origin preserves differences but changes raw-value ratios.

**Next:** [1.4 Variables](4_Variables.ipynb).